In [1]:
import os, re, string, numpy as np, pandas as pd

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"




In [2]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))




/kaggle/input/train.csv
/kaggle/input/test.csv
/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv.zip
/kaggle/input/test.csv.zip
/kaggle/input/sample_submission.csv.zip
/kaggle/input/tweet-sentiment-extraction/train.csv
/kaggle/input/tweet-sentiment-extraction/test.csv
/kaggle/input/tweet-sentiment-extraction/description.md
/kaggle/input/tweet-sentiment-extraction/sample_submission.csv
/kaggle/input/tweet-sentiment-extraction/train.csv.zip
/kaggle/input/tweet-sentiment-extraction/test.csv.zip
/kaggle/input/tweet-sentiment-extraction/sample_submission.csv.zip


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout




2026-05-07 10:30:51.700567: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778149851.714986      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778149851.719337      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 10:30:51.737299: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
train_path = "../input/tweet-sentiment-extraction/train.csv"
data = pd.read_csv(train_path)




In [5]:
print("Rows:", len(data), "Unique IDs:", len(data.textID.unique()))
print("Sample row:", data.head(1))




Rows: 24732 Unique IDs: 24732
Sample row:        textID                                               text  \
0  8d4ad58b45  eating breakfast  getting ready to go to schoo...   

                                       selected_text sentiment  
0  eating breakfast  getting ready to go to schoo...  negative  


In [6]:
data = data[pd.notnull(data.selected_text)].reset_index(drop=True)




In [7]:
def clean_series(series):
    cleaned = []
    for txt in series:
        if isinstance(txt, str):
            txt = txt.lower()
            txt = re.sub(r"^https?://\S+", "", txt)
            txt = txt.strip()
        cleaned.append(txt)
    return cleaned


# apply the same cleaning to training texts and selected_texts
data["text"] = clean_series(data["text"])
data["selected_text"] = clean_series(data["selected_text"])




In [8]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    data, test_size=0.25, random_state=42, stratify=data.sentiment
)




In [9]:
vocab_size = 10000
oov_tok = "<OOV>"
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok, lower=False, split=" ")
tokenizer.fit_on_texts(train_df.text)
word_index = tokenizer.word_index




In [10]:
max_len = 50
trunc_type = "post"
pad_type = "post"


def texts_to_padded(texts, maxlen=None):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=maxlen, truncating=trunc_type, padding=pad_type)


train_x = texts_to_padded(train_df.text, maxlen=max_len)
val_x = texts_to_padded(val_df.text, maxlen=max_len)


def texts_to_mask(texts, selected_texts):
    masks = []
    for txt, sel in zip(texts, selected_texts):
        txt_seq = tokenizer.texts_to_sequences([txt])[0]
        sel_seq = set(tokenizer.texts_to_sequences([sel])[0])
        mask = [1 if token in sel_seq else 0 for token in txt_seq]
        if len(mask) < max_len:
            mask = mask + [0] * (max_len - len(mask))
        else:
            mask = mask[:max_len]
        masks.append(mask)
    return np.array(masks)


train_y = texts_to_mask(train_df.text, train_df.selected_text)
val_y = texts_to_mask(val_df.text, val_df.selected_text)




In [11]:
rev_word_index = {v: k for k, v in word_index.items()}


def decode_phrase(padded_seq, mask_seq):
    """Return a space‑separated string of tokens where mask==1."""
    indices = padded_seq[mask_seq.astype(bool)]
    words = [rev_word_index.get(i, "") for i in indices if i != 0]
    return " ".join(words).strip()




In [12]:
def build_model():
    model = Sequential(
        [
            Embedding(vocab_size, 16, input_length=max_len),
            Dropout(0.5),
            Bidirectional(LSTM(20)),
            Dropout(0.5),
            Dense(max_len, activation="sigmoid"),
        ]
    )
    loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    model.compile(
        loss=loss_fn, optimizer=tf.keras.optimizers.Adam(0.001), metrics=["accuracy"]
    )
    return model


positive_model = build_model()
negative_model = build_model()




/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-07 10:30:59.691250: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778149859.691817      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


In [13]:
pos_train_mask = train_y[train_df.sentiment == "positive"]
pos_val_mask = val_y[val_df.sentiment == "positive"]
pos_train_x = train_x[train_df.sentiment == "positive"]
pos_val_x = val_x[val_df.sentiment == "positive"]

positive_history = positive_model.fit(
    pos_train_x.astype(float),
    pos_train_mask.astype(float),
    epochs=20,
    batch_size=32,
    verbose=2,
    validation_data=(pos_val_x.astype(float), pos_val_mask.astype(float)),
)




Epoch 1/20


I0000 00:00:1778149863.277043     106 cuda_dnn.cc:529] Loaded cuDNN version 90300


182/182 - 5s - 29ms/step - accuracy: 0.1239 - loss: 0.3220 - val_accuracy: 0.2947 - val_loss: 0.2016
Epoch 2/20
182/182 - 1s - 5ms/step - accuracy: 0.1579 - loss: 0.2028 - val_accuracy: 0.3697 - val_loss: 0.1902
Epoch 3/20
182/182 - 1s - 5ms/step - accuracy: 0.1800 - loss: 0.1904 - val_accuracy: 0.2983 - val_loss: 0.1837
Epoch 4/20
182/182 - 1s - 6ms/step - accuracy: 0.1986 - loss: 0.1840 - val_accuracy: 0.3599 - val_loss: 0.1825
Epoch 5/20
182/182 - 1s - 6ms/step - accuracy: 0.2231 - loss: 0.1781 - val_accuracy: 0.3697 - val_loss: 0.1839
Epoch 6/20
182/182 - 1s - 7ms/step - accuracy: 0.2491 - loss: 0.1745 - val_accuracy: 0.3526 - val_loss: 0.1845
Epoch 7/20
182/182 - 1s - 6ms/step - accuracy: 0.2650 - loss: 0.1698 - val_accuracy: 0.3402 - val_loss: 0.1830
Epoch 8/20
182/182 - 1s - 6ms/step - accuracy: 0.2798 - loss: 0.1660 - val_accuracy: 0.3180 - val_loss: 0.1833
Epoch 9/20
182/182 - 1s - 6ms/step - accuracy: 0.2837 - loss: 0.1627 - val_accuracy: 0.3180 - val_loss: 0.1847
Epoch 10/20

In [14]:
neg_train_mask = train_y[train_df.sentiment == "negative"]
neg_val_mask = val_y[val_df.sentiment == "negative"]
neg_train_x = train_x[train_df.sentiment == "negative"]
neg_val_x = val_x[val_df.sentiment == "negative"]

negative_history = negative_model.fit(
    neg_train_x.astype(float),
    neg_train_mask.astype(float),
    epochs=20,
    batch_size=32,
    verbose=2,
    validation_data=(neg_val_x.astype(float), neg_val_mask.astype(float)),
)




Epoch 1/20
165/165 - 3s - 21ms/step - accuracy: 0.0901 - loss: 0.3454 - val_accuracy: 0.1566 - val_loss: 0.2180
Epoch 2/20
165/165 - 1s - 6ms/step - accuracy: 0.1160 - loss: 0.2280 - val_accuracy: 0.0886 - val_loss: 0.2052
Epoch 3/20
165/165 - 1s - 6ms/step - accuracy: 0.1116 - loss: 0.2134 - val_accuracy: 0.1257 - val_loss: 0.1979
Epoch 4/20
165/165 - 1s - 6ms/step - accuracy: 0.1130 - loss: 0.2043 - val_accuracy: 0.0863 - val_loss: 0.1930
Epoch 5/20
165/165 - 1s - 6ms/step - accuracy: 0.1164 - loss: 0.1987 - val_accuracy: 0.0851 - val_loss: 0.1906
Epoch 6/20
165/165 - 1s - 6ms/step - accuracy: 0.1170 - loss: 0.1933 - val_accuracy: 0.0846 - val_loss: 0.1906
Epoch 7/20
165/165 - 1s - 6ms/step - accuracy: 0.1158 - loss: 0.1858 - val_accuracy: 0.0823 - val_loss: 0.1910
Epoch 8/20
165/165 - 1s - 6ms/step - accuracy: 0.1143 - loss: 0.1814 - val_accuracy: 0.0834 - val_loss: 0.1966
Epoch 9/20
165/165 - 1s - 6ms/step - accuracy: 0.0989 - loss: 0.1774 - val_accuracy: 0.0834 - val_loss: 0.1953


In [15]:
test_path = "../input/tweet-sentiment-extraction/test.csv"
test = pd.read_csv(test_path)




In [16]:
test["text"] = clean_series(test["text"])




In [17]:
test_x = texts_to_padded(test.text, maxlen=max_len)




In [18]:
preds_masks = []
for idx, row in test.iterrows():
    seq = test_x[idx][np.newaxis]  # shape (1, max_len)
    if row.sentiment == "positive":
        pred = np.round(positive_model.predict(seq))
    elif row.sentiment == "negative":
        pred = np.round(negative_model.predict(seq))
    else:  # neutral – use full text later
        pred = np.zeros_like(seq)
    preds_masks.append(pred.squeeze())

preds_masks = np.array(preds_masks)




1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━

In [19]:
selected_texts = []
for i, row in test.iterrows():
    if row.sentiment == "neutral":
        selected_texts.append(row.text)
    else:
        mask = preds_masks[i]
        phrase = decode_phrase(test_x[i], mask)
        selected_texts.append(phrase if phrase else row.text)

test["prediction"] = selected_texts




In [20]:
submission = test[["textID", "prediction"]].rename(
    columns={"prediction": "selected_text"}
)
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
